<a href="https://colab.research.google.com/github/Shiveshrane/Research_paper_implementations/blob/main/StableDiffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
from tqdm import tqdm

# VAE

## Required blocks

In [ ]:
class SelfAttention(tf.keras.layers.Layer):
  def __init__(self, n_heads:int, d_embed:int, in_proj_bias=True, out_proj_bias=True):
      super().__init__()
      assert d_embed%n_heads==0, "d_embed must be divisible by n_heads"
      self.n_heads=n_heads
      self.d_embed=d_embed
      self.d_head=int(d_embed/n_heads)
      self.in_proj=layers.Dense(3*d_embed, use_bias=in_proj_bias) # We create a single model
      self.out=layers.Dense(d_embed, use_bias=out_proj_bias)
  def call(self, x:tf.Tensor, causal_mask=False)->tf.Tensor:
      #X=(B,S,D)
      batch_size, seq_len, d_embed=x.shape
      intermim_shape=(batch_size, seq_len, self.n_heads, self.d_head)
      qkv=self.in_proj(x) # (B,S,3*D)
      q,k,v=tf.split(qkv, 3, axis=-1) # (B,S,D) *3
      q=tf.reshape(q, intermim_shape)
      k=tf.reshape(k, intermim_shape)
      v=tf.reshape(v, intermim_shape)

      # (B,S,D)=> (B,S,H,D/H)=>(B,H,S,D/H)
      q=tf.transpose(q, perm=(0,2,1,3))
      k=tf.transpose(k, perm=(0,2,1,3))
      v=tf.transpose(v, perm=(0,2,1,3))

      attention=tf.matmul(q, k, transpose_b=True)/tf.sqrt(tf.cast(self.d_head, tf.float32))
      if causal_mask:
        mask=tf.linalg.band_part(tf.ones((seq_len, seq_len), dtype=tf.bool), -1,0)
        mask=tf.reshape(mask, (1,1,seq_len, seq_len))
        attention=tf.where(mask, attention, -np.inf)

      attention=tf.nn.softmax(attention, axis=-1)
      out=tf.matmul(attention, v)
      out=tf.transpose(out, perm=(0,2,1,3))
      out=tf.reshape(out, (batch_size, seq_len, d_embed))
      out=self.out(out)
      return out



In [ ]:
class VAE_Residual_Block(tf.keras.layers.Layer):
  def __init__(self, channels: int):
    super(VAE_Residual_Block, self).__init__()
    self.channels=channels
    self.groupnorm1=layers.GroupNormalization(groups=32)
    self.conv1=layers.Conv2D(channels, kernel_size=3, padding="same")
    self.groupnorm2=layers.GroupNormalization(groups=32)
    self.conv2=layers.Conv2D(channels, kernel_size=3, padding="same")

    self.skip=None

  def build(self, input_shape):
    in_channels=input_shape[-1]
    if in_channels==self.channels:
      self.skip=lambda x: x
    else:
      self.skip=layers.Conv2D(self.channels, kernel_size=1, padding="same")


  def call(self, x: tf.Tensor) -> tf.Tensor:
      # X: (B,H,W,C)
      residual=x
      x=self.groupnorm1(x)
      x=tf.nn.silu(x)
      x=self.conv1(x)
      x=self.groupnorm2(x)
      x=tf.nn.silu(x)
      x=self.conv2(x)
      x+=self.skip(residual)
      return x

In [ ]:
class VAE_Attention_Block(tf.keras.layers.Layer):
  def __init__(self, channels):
    super(VAE_Attention_Block, self).__init__()
    self.groupnorm=layers.GroupNormalization(groups=32)
    self.attention=SelfAttention(1, channels)

  def call(self, x: tf.Tensor) -> tf.Tensor:
    # x: (B,H,W,C)
    residual=x
    b,h,w,c=x.shape

    x=tf.reshape(x, shape=(b,h*w,c))
   #x=tf.transpose(x, perm=(-1,-2))
    x=self.attention(x)
    x=tf.reshape(x, shape=(b,h,w,c))
   #x=self.groupnorm(x)
    x+=residual
    return x


## Encoder

In [ ]:
class VAE_encoder(tf.keras.Sequential):
  def __init__(self):
    super(VAE_encoder, self).__init__([
        #(B,H,W,C)==> (B,H,W,128)
        tf.keras.layers.Conv2D(128, kernel_size=3, padding="same"),

        # (B,H,W,128)==> (B,H,W,128)
        VAE_Residual_Block(128),

        # (B,H,W,128)===> (B,128, H/2, W/2)
        tf.keras.layers.Conv2D(128, kernel_size=3, strides=2, padding="valid"),

        # (B,H/2,W/2,128)==> (B,H/2,W/2,256)
        VAE_Residual_Block(256),

        # (B,H/2,W/2,256)=> (B,H/2,W/2,256)
        VAE_Residual_Block(256),

        #(B,H/4,W/4, 256)
        tf.keras.layers.Conv2D(256, kernel_size=3, strides=2, padding="valid"),

        # (B,H/4,W/4,256)=> (B,H/4,W/4,512)
        VAE_Residual_Block(512),

        # (B,H/4,W/4,512)=> (B,H/4,W/4,512)
        VAE_Residual_Block(512),

        # (B,H/4,W/4,512)=> (B,H/8,W/8,512)
        tf.keras.layers.Conv2D(512, kernel_size=3, strides=2, padding="valid"),

        VAE_Residual_Block(512),
        # (B,H/8,W/8,512)=> (B,H/8,W/8,512)
        VAE_Residual_Block(512),

        # (B,H/8,W/8,512)=> (B,H/8,W/8,512)
        VAE_Residual_Block(512),

        # (B,H/8,W/8,512)=> (B,H/8,W/8,512)
        VAE_Attention_Block(512),

        # (B,H/8,W/8,512)=> (B,H/8,W/8,512)
        VAE_Residual_Block(512),


        layers.GroupNormalization(32),

        tf.keras.layers.Lambda(lambda x: tf.nn.silu(x)),

        layers.Conv2D(8, kernel_size=3, padding="same"),
        layers.Conv2D(8, kernel_size=1, padding="valid")





    ])

  def call(self, x: tf.Tensor, noise: tf.Tensor) -> tf.Tensor:
    # x: (B,H,W,C)
    #noise (B, H/8, W/8,C)
    for module in self:
      if getattr(module, "strides", None)==(2,2):
        x=tf.pad(x, paddings=[[0,0], [0,1], [0,1], [0,0]])
      x=module(x)

    mean, log_variance=tf.split(x, 2, axis=-1) # (B,H/8, W/8, C)--> two tensors of (B,H/8, W/8, C/2)
    log_variance=tf.clip_by_value(log_variance, -30.0, 20.0) # (B,H/8, W/8, C/2)
    variance=tf.exp(log_variance)
    std_dev=tf.sqrt(variance)
    #z=N(0,1)--> N(mean, variance)?
    # x=mean+std_dev*z--> This is how we can convert one gaussian to anothe, given mean and variance of resultant gaussian.
    x=mean+std_dev*noise

    # Scale the output using constant
    x*=0.18215
    return x



## Decoder

In [ ]:
class Decoder(tf.keras.Sequential):
  def __init__(self):
    super(Decoder, self).__init__([
            layers.Conv2D(4, kernel_size=1, padding="valid"),
            layers.Conv2D(512, kernel_size=3, padding="same"),
            VAE_Residual_Block(512),
            VAE_Attention_Block(512),
            VAE_Residual_Block(512),
            VAE_Residual_Block(512),
            VAE_Residual_Block(512),
            VAE_Residual_Block(512),
            layers.UpSampling2D(size=2, interpolation='nearest'),
            layers.Conv2D(512, kernel_size=3, padding="same"),
            VAE_Residual_Block(512),
            VAE_Residual_Block(512),
            VAE_Residual_Block(512),
            layers.UpSampling2D(size=2, interpolation='nearest'),
            layers.Conv2D(512, kernel_size=3, padding="same"),
            VAE_Residual_Block(256),
            VAE_Residual_Block(256),
            VAE_Residual_Block(256),
            layers.UpSampling2D(size=2, interpolation='nearest'),
            layers.Conv2D(256, kernel_size=3, padding="same"),
            VAE_Residual_Block(128),
            VAE_Residual_Block(128),
            VAE_Residual_Block(128),
            layers.GroupNormalization(groups=32),
            layers.Lambda(lambda x: tf.nn.silu(x)),
            layers.Conv2D(3, kernel_size=3, padding="same")

    ])
  def call(self, x:tf.Tensor)->tf.Tensor:
    # x=(B,H/8, W/8, 4)
    x/=0.18215
    for module in self:
      x=module(x)
    return x

# CLIP Encoder

In [ ]:
def QuickGelu(x:tf.Tensor):
  return x*tf.nn.sigmoid(1.702*x)

In [ ]:
class CLIPEmbedding(tf.keras.layers.Layer):
  def __init__(self, vocab_size:int, embed_dim:int, num_tokens:int):
    super().__init__()
    self.token_embedding=tf.keras.layers.Embedding(vocab_size, embed_dim, name="token_embedding")
    self.position_embedding=tf.keras.layers.Embedding(num_tokens, embed_dim, name="position_embedding")
  def call(self,tokens ):
    #(B,S)-> (B,S,D)
    x=self.token_embedding(tokens)
    x+=self.position_embedding(tokens)
    return x

In [ ]:
class CLIPLayer(tf.keras.layers.Layer):
  def __init__(self, n_heads:int, d_embed:int):
    super().__init__()
    self.norm1=tf.keras.layers.LayerNormalization(d_embed)
    self.attn=SelfAttention(n_heads, d_embed)
    self.norm2=tf.keras.layers.LayerNormalization(d_embed)
    self.linear1=tf.keras.layers.Dense(d_embed*4)
    self.linear2=tf.keras.layers.Dense(d_embed)

  def call(self, x:tf.Tensor)->tf.Tensor: # Transformer block
    #(B,S,D)
    residue=x
    ## Self attention
    x=self.norm1(x)
    x=self.attn(x, causal_mask=True)
    x+=residue
    residue=x
    x=self.norm2(x)
    x=self.linear1(x)
    x= QuickGelu(x) # Quick GELU activation
    x=self.linear2(x)
    x+=residue
    return x





In [ ]:
class Clip(tf.keras.layers.Layer):
  def __init__(self):
    self.embedding=CLIPEmbedding(49408, 768, 77)
    self.layers=[
        CLIPLayer(12,768) for i in range(12)
    ]
    self.layernorm=tf.keras.layers.LayerNormalization(768)
  def call(self, tokens: tf.Tensor)-> tf.Tensor:
    tokens=tf.cast(tokens, tf.int64)
    # (B,S)--> (B,S,D)
    state=self.embedding(tokens)
    for layer in self.layers:
      state=layer(state)
    output=self.layernorm(state)
    return output

# UNET

## Switch Sequential

In [ ]:
class SwitchSequential(tf.keras.Sequential):
  def call(self,x: tf.Tensor, context:tf.Tensor, time:tf.Tensor):
    for layer in self:
      if isinstance(layer, UNET_AttentionBlock):
        x=layer(x, context)
      elif isinstance(layer, UNET_ResidualBlock):
        x=layer(x, time)
      else:
        x=layer(x)
    return x

## Upsample

In [ ]:
class UpSample(tf.keras.layers.Layer):
  def __init__(self, channels):
    super().__init__()
    self.conv=tf.keras.layers.Conv2D(channels, kernel_size=3, padding="same")
    self.upsample=tf.keras.layers.UpSampling2D(size=2, interpolation="nearest")
  def call(self, x):
    x=self.upsample(x)
    x=self.conv(x)
    return x

## UNET Output Layer

In [ ]:
class UNET_OutputLayer(tf.keras.layers.Layer):
  def __init__(self, in_channels:int, out_channels:int):
    super().__init__()
    self.groupnorm=tf.keras.layers.GroupNormalization(groups=32)
    self.conv=tf.keras.layers.Conv2D(out_channels, kernel_size=3, padding="same")
  def call(self, x:tf.Tensor):
    #(B,H/8, W/8, 320)-> (B,H/4,W/4,4)
    x=self.groupnorm(x)
    x=tf.nn.silu(x)
    x=self.conv(x)
    return x

## UNET Residual Block

In [ ]:
class UNET_ResidualBlock(tf.keras.layers.Layer):
  def __init__(self, in_channels:int, out_channels:int, n_time=1280):
    super().__init__()
    self.groupnorm_feature=tf.keras.layers.GroupNormalization(32)
    self.conv_feature=tf.keras.layers.Conv2D(out_channels, kernel_size=3, padding="same")
    self.linear_time=tf.keras.layers.Dense(out_channels)
    self.groupnorm_merged=tf.keras.layers.GroupNormalization(32)
    self.conv_merged=tf.keras.layers.Conv2D(out_channels, kernel_size=3, padding="same")

    if in_channels==out_channels:
      self.skip=lambda x: x
    else:
      self.skip=tf.keras.layers.Conv2D(out_channels, kernel_size=1, padding="same")
  def call(self, feature, time):
    # (B,H,W,IN_C)
    #time (1,1280)
    residual=feature
    feature=self.groupnorm_feature(feature)
    feature=tf.nn.silu(feature)
    feature=self.conv_feature(feature)

    time=tf.nn.silu(time)
    time=self.linear_time(time)
    time=tf.nn.silu(time)
    time=tf.expand_dims(time, axis=1)
    time=tf.expand_dims(time, axis=1)
    merged=feature+time
    merged=self.groupnorm_merged(merged)
    merged=tf.nn.silu(merged)
    merged=self.conv_merged(merged)
    merged+=self.skip(residual)
    return merged

## UNET Attention Block

In [ ]:
class CrossAttention(tf.keras.layers.Layer):
  def __init__(self, n_heads:int, d_embd:int, d_cross:int, in_proj_bias=True, out_proj_bias=True):
    super().__init__()
    self.n_heads=n_heads
    self.n_embd=d_embd
    self.d_cross=d_cross
    self.q_proj=tf.keras.layers.Dense(d_embd, use_bias=in_proj_bias)
    self.k_proj=tf.keras.layers.Dense(d_embd, use_bias=in_proj_bias)
    self.v_proj=tf.keras.layers.Dense(d_embd, use_bias=in_proj_bias)
    self.out_proj=tf.keras.layers.Dense(d_embd, use_bias=out_proj_bias)
    self.d_head=int(d_embd/n_heads)

  def call(self, x, y):
    #x: (latent): (Batch_size, seq_len, DimQ)
    # context:y: (batch_size, seq_len_kv, dim_kv) =(B,77,768)

    batch_size, sequence_len, d_embd=x.shape
    interim_shape=(batch_size, sequence_len, self.n_heads, self.d_head)
    q=self.q_proj(x)
    k=self.k_proj(y)
    v=self.v_proj(y)
    q=tf.reshape(q, interim_shape)
    k=tf.reshape(k, interim_shape)
    v=tf.reshape(v, interim_shape)

    q=tf.transpose(q, perm=(0,2,1,3))
    k=tf.transpose(k, perm=(0,2,1,3))
    v=tf.transpose(v, perm=(0,2,1,3))

    attention=tf.matmul(q, k, transpose_b=True)/tf.sqrt(tf.cast(self.d_head, tf.float32))
    attention=tf.nn.softmax(attention, axis=-1)
    out=tf.matmul(attention, v)
    out=tf.transpose(out, perm=(0,2,1,3))
    out=tf.reshape(out, (batch_size, sequence_len, d_embd))
    out=self.out_proj(out)
    return out


In [ ]:
class UNET_AttentionBlock(tf.keras.layers.Layer):
  def __init__(self, n_heads:int, n_embd:int, d_context=768):
    super().__init__()
    channels=n_heads*n_embd
    self.groupnorm=tf.keras.layers.GroupNormalization(groups=32, epsilon=1e-6)
    self.conv_input=tf.keras.layers.Conv2D(channels, kernel_size=1, padding="same")
    self.attention=SelfAttention(n_heads, n_embd)
    self.layernorm1=tf.keras.layers.LayerNormalization()
    self.layernorm2=tf.keras.layers.LayerNormalization()
    self.layernorm3=tf.keras.layers.LayerNormalization()
    self.attention1=SelfAttention(n_heads, channels, in_proj_bias=False)
    self.attention2=CrossAttention(n_heads, channels, in_proj_bias=False)
    self.linear_geglu1=tf.keras.layers.Dense(4*channels*2)
    self.linear_geglu2=tf.keras.layers.Dense(channels)
    self.conv_output=tf.keras.layers.Conv2D(channels, kernel_size=1, padding="same")

  def call(self,x, context):

      # (Batch_size, features, height, width)
      # context: (Batch_size, Seq_len, dim)
      residue_long=x
      x=self.groupnorm(x)
      x=self.conv_input(x)
      B,H,W,C=x.shape
      # Normalize +self attention with skip connection
      x=tf.reshape(x, shape=(B,H*W,C)) #(B,H,W,C)==>(B,H*W,C)
      residual_short=x
      x=self.layernorm1(x)
      x=self.attention(x)
      x=x+residual_short
      residual_short=x
      # Normalization+Cross attention
      x=self.layernorm2(x)
      x=self.attention2(x, context)
      x=x+residual_short
      residual_short=x

      # Normalization + FF with GeGlu and skip connections
      x=self.layernorm3(x)
      x=self.linear_geglu1(x)
      x,gate=tf.split(x,2,axis=-1)
      x=x*tf.nn.gelu(gate)
      x=self.linear_geglu2(x)
      x=x+residual_short

      x=tf.reshape(x, shape=(B,H,W,C))
      x=self.conv_output(x)+residue_long
      return x




## UNET

In [ ]:
class UNET(tf.keras.layers.Layer):
  def __init__(self):
    super().__init__()
    self.encoders=[
        SwitchSequential(tf.keras.layers.Conv2D(320, kernel_size=3, padding="same")),
        SwitchSequential(UNET_ResidualBlock(320,320),
                         UNET_AttentionBlock(8,40)),
        SwitchSequential(UNET_ResidualBlock(320,320),
                         UNET_AttentionBlock(8,40)),
        SwitchSequential(tf.keras.layers.Conv2D(320, kernel_size=3, strides=2, padding="same")),
        SwitchSequential(UNET_ResidualBlock(320,640),
                         UNET_AttentionBlock(8,80)),
         SwitchSequential(UNET_ResidualBlock(640,640),
                         UNET_AttentionBlock(8,80)),
        SwitchSequential(tf.keras.layers.Conv2D(640, kernel_size=3, strides=2, padding="same")),
        SwitchSequential(UNET_ResidualBlock(640,1280),
                         UNET_AttentionBlock(8,160)),
        SwitchSequential(UNET_ResidualBlock(1280,1280),
                         UNET_AttentionBlock(8,160)),
        SwitchSequential(tf.keras.layers.Conv2D(1280, kernel_size=3, strides=2, padding="same")),
        SwitchSequential(UNET_ResidualBlock(1280,1280)),
        SwitchSequential(UNET_ResidualBlock(1280,1280)),
    ]

    self.bottle_neck=tf.keras.Sequential([
      UNET_ResidualBlock(1280,1280),
      UNET_AttentionBlock(8,160),
      UNET_ResidualBlock(1280,1280)
    ])
    self.decoders= [
        SwitchSequential(UNET_ResidualBlock(2560,1280)),
        SwitchSequential(UNET_ResidualBlock(2560,1280)),
        SwitchSequential(UNET_ResidualBlock(2560, 1280), UpSample(1280)),

        SwitchSequential(UNET_ResidualBlock(2560,1280), UNET_AttentionBlock(8,160)),
        SwitchSequential(UNET_ResidualBlock(2560,1280), UNET_AttentionBlock(8,160)),
        SwitchSequential(UNET_ResidualBlock(1920, 1280), UNET_AttentionBlock(8,160), UpSample(1280)),
        SwitchSequential(UNET_ResidualBlock(1920, 640), UNET_AttentionBlock(8,80)),#
       # SwitchSequential(UNET_ResidualBlock(1280, 640), UNET_AttentionBlock(8,80)),
        SwitchSequential(UNET_ResidualBlock(1280, 640), UNET_AttentionBlock(8,80)),
        SwitchSequential(UNET_ResidualBlock(960, 640), UNET_AttentionBlock(8,80), UpSample(640)),
        SwitchSequential(UNET_ResidualBlock(960, 320), UNET_AttentionBlock(8,40)),
        SwitchSequential(UNET_ResidualBlock(640, 320), UNET_AttentionBlock(8,80)),
        SwitchSequential(UNET_ResidualBlock(640, 320), UNET_AttentionBlock(8,40)),
    ]

  def call(self, x:tf.Tensor, context:tf.Tensor, time:tf.Tensor):
    skip_connections=[]
    for layers in self.encoders:
      x=layers(x, context, time)
      skip_connections.append(x)
    x=self.bottle_neck(x)
    for layers in self.decoders:
      x=tf.concat([x, skip_connections.pop()], axis=-1)
      x=layers(x, context, time)
    return x

# TimeEmbedding

In [ ]:
class TimeEmbedding(tf.keras.layers.Layer):
  def __init__(self, n_embd:int):
    super().__init__()
    self.n_embd=n_embd
    self.linear1=tf.keras.layers.Dense(n_embd*4)
    self.linear2=tf.keras.layers.Dense(4*n_embd)
  def call(self,time:tf.Tensor):
    x=self.linear1(time)
    x=tf.nn.silu(x)
    x=self.linear2(x)
    return x


# Diffusion

In [ ]:
class Diffusion(tf.keras.Model):
  def __init__(self):
    self.time_Embedding=TimeEmbedding(320)
    self.unet=UNET()
    self.final=UNET_OutputLayer(320,4)

  def call(self, latent:tf.Tensor, context: tf.Tensor, time:tf.Tensor):
    #latent-> (B,H/8,W/8,4)
    #context-> (B,S,D)
    #time (1,320)

    time=self.time_Embedding(time) # Its like positional embedding like positonal Encoding, like in LLMs
    output=self.unet(latent, context, time) #(B,H/8,W/8,4)--> (B,H/8,W/8,320)
    output=self.final(output) # (B,H/8,W/8,320)--> (B,H/8,W/8,4)
    return output


# DDPM Scheduler

In [ ]:
class DDPMSampler(tf.keras.layers.Layer):
  def __init__(self, num_training_steps=1000, beta_start=0.00085, beta_end=0.0128):
    self.betas=tf.linspace(beta_start**0.5, beta_end**0.5, num_training_steps, dtype=tf.float32)**2
    self.alpha=1.0-self.betas
    self.alpha_cumprod=tf.math.cumprod(self.alphas, 0)
    self.one=tf.constant(1.0, dtype=tf.float32)
    self.num_training_steps=num_training_steps
    self.timesteps=np.arange(0, num_training_steps)[::-1].astype(np.int64)
    self.timesteps=tf.convert_to_tensor(self.timesteps, dtype=tf.int64)

  def set_inference_steps(self, num_inference_steps=50):
    self.num_inference_steps=num_inference_steps
    step_ratio=self.num_training_steps//self.num_inference_steps
    timesteps=(np.arange(0, num_inference_steps)*step_ratio).round()[::-1].copy().astype(np.int64)
    self.timesteps=tf.convert_to_tensor(timesteps, dtype=tf.int64)


# Forward process
  def add_noise(self, original_sample, timesteps):
    alpha_cumprod=self.alpha_cumprod
    sqrt_alpha_cumprod=tf.sqrt(tf.gather(alpha_cumprod, timesteps))
    sqrt_alpha_cumprod= tf.reshape(sqrt_alpha_cumprod, [-1]+[1]*(len(original_sample.shape)-1))
    sqrt_1_minus_alpha_prod=tf.sqrt(1-tf.gather(alpha_cumprod, timesteps))
    sqrt_1_minus_alpha_prod= tf.reshape(sqrt_1_minus_alpha_prod, [-1]+[1]*(len(original_sample.shape)-1))

    noise=tf.random.normal(original_sample.shape, dtype=original_sample.dtype)
    noisy_sample= sqrt_alpha_cumprod*original_sample+sqrt_1_minus_alpha_prod*noise
    return noisy_sample

  def get_previous_timestep(self, timestep):
    return timestep-(self.num_training_steps //self.num_inference_steps)



  def get_variance(self, timestep_t):
    prev_t=self.get_previous_timestep(timestep_t)
    alpha_prod_t=tf.gather(self.alpha_cumprod, timestep_t)
    alpha_prod_t_prev=tf.gather(self.alpha_cumprod, prev_t) if prev_t>=0 else self.one
    beta_prod_t=1-alpha_prod_t
    beta_prod_t_prev=1-alpha_prod_t_prev
    current_beta_t=1-alpha_prod_t/alpha_prod_t_prev

    ##Variance
    variance=(beta_prod_t_prev*current_beta_t)/beta_prod_t
    variance=tf.clip_by_value(variance, 1e-20, 1)
    return variance #Formula 7


  # Reverse process single step
  def step(self, timestep, latents, model_output):
    t=timestep
    prev_t=self.get_previous_timestep(t)
    alpha_prod_t=tf.gather(self.alpha_cumprod, t)
    alpha_prod_t_prev=tf.gather(self.alpha_cumprod,prev_t) if prev_t>=0 else self.one
    beta_prod_t= 1-alpha_prod_t
    beta_prod_t_prev=1-alpha_prod_t_prev
    current_alpha_t=alpha_prod_t/alpha_prod_t_prev
    current_beta_t=1-current_alpha_t

    x0=(latents-tf.sqrt(beta_prod_t)*model_output)/tf.sqrt(alpha_prod_t) ## computing original sample using formula 15 of DDPM paper

    #mean_rev=(tf.sqrt(alpha_prod_t_prev)*current_beta_t/(1-alpha_prod_t))*x0 + ((tf.sqrt(current_alpha_t)*beta_prod_t_prev)/(beta_prod_t))


    ## Mean
    original_sample_coeff=tf.sqrt(alpha_prod_t_prev)*current_beta_t/(1-alpha_prod_t)
    current_sample_coeff=(tf.sqrt(current_alpha_t)*beta_prod_t_prev)/(beta_prod_t)
    mean_rev= original_sample_coeff*x0+current_sample_coeff*latents #Formula 7

    #Variance
    variance_rev=0
    if t>0:
      variance_rev=self.get_variance(t)
      noise=tf.random.randn(model_output.shape, dtype= model_output.dtype)
      variance=tf.sqrt(variance_rev)*noise

    pred_pred_sample= mean_rev+variance
    return pred_pred_sample

  def set_strength(self, strength=1):
    start_step=self.num_inference_steps-int(self.num_inference_steps*strength)
    self.timesteps=self.timesteps[start_step:]
    self.start_step=start_step






# DDIM Scheduler

# Pipeline

In [ ]:
WIDTH=512
HEIGHT=512
LATENT_WIDTH=512//8
LATENT_HEIGHT=HEIGHT//8

In [ ]:
def generate(prompt:str, uncond_prompt:str, input_img=None, strength=0.0, do_cfg=True, cfg_scale=7.5,
             sampler_name="ddpm", n_inference_steps=50, models={}, seed=None, tokenizer=None
             ):
          # cfg scale is the weight of the Classifier free guidance. which is
          if not(0< strength<=1):
            raise ValueError("Strength mus tbe btw 0 and 1")
          generator=tf.random.Generator()
          if seed is None:
            tf.random.set_seed(np.random.randint(0, np.inf))
          else:
            tf.random.set_seed(42)
          clip=models["clip"]
          #clip.to(device)
          if do_cfg:
            # convert the prompt to token using tokenizer
            cond_tokens=tokenizer.batch_encode_plus([prompt], padding="max_length", max_length=77, return_tensors="tf").input_ids
            # convert to tensor to (B,S)
            cond_tokens=tf.convert_to_tensor(cond_tokens, dtype=tf.int64)
            # (B,S)==> (B,S,768)
            cond_context=clip(cond_tokens)
            uncond_tokens=tokenizer.batch_encode_plus([uncond_prompt], padding="max_length", max_length=77, return_tensors="tf").input_ids
            uncond_tokens=tf.convert_to_tensor(uncond_tokens, dtype=tf.int64)
            uncond_context=clip(uncond_tokens)
            context=tf.concat([uncond_context, cond_context], axis=0)
          else:
            cond_tokens=tokenizer.batch_encode_plus([prompt], padding="max_length", max_length=77, return_tensors="tf").input_ids
            cond_tokens=tf.convert_to_tensor(cond_tokens, dtype=tf.int64)
            context=clip(cond_tokens)
          if sampler_name=="ddpm":
            sampler=DDPMSampler()
            sampler.set_inference_steps(n_inference_steps)
          else:
            raise ValueError(f"Unknown sampler {sampler_name}")

          latent_shape=(1, LATENT_HEIGHT, LATENT_WIDTH, 4)

          if input_img:
            encoder=models["encoder"]
            #input_img_tensor=input_img.resize(input_img, size=(WIDTH, HEIGHT))
            input_img_tensor=input_img.resize((WIDTH, HEIGHT))
            input_img_tensor=np.array(input_img_tensor)
            input_img_tensor=tf.convert_to_tensor(input_img_tensor, dtype=tf.float32)
            input_img_tensor=rescale(input_img_tensor, (0,255), (-1,1))
            input_img_tenso=tf.expand_dims(input_img_tensor, axis=0) # (B,H,W,C)
            # In tensorflow, the actual way of doing operations on image is (B,H,W,C),
            #so unlike in pytorch version, we directly pass the image to the Encoder.
            encoder_noise=tf.random.normal(shape=latent_shape, dtype=tf.float32)
            #run the image through VAE encoder to get the latent probabilities
            latents=encoder(input_img_tensor, encoder_noise)
            sampler.set_strength(strength=strength)
            latents=sampler.add_noise(latents,sampler.timesteps[0])

          else:
            latents=tf.random.normal(shape=latent_shape, dtype=tf.float32)

          diffusion=models["diffusion"]

          timesteps=tqdm(sampler.timesteps)
          for i, timestep in enumerate(timesteps):
            time_embedding=get_time_embedding(timestep)
            model_input=latent

            if do_cfg:
              model_input= tf.repeat(model_input, repeats=2, axis=0)

              # Model output is predicted by UNET
              model_output=diffusion(model_input, context, time_embedding)

            if do_cfg:
              output_cond, output_uncond=tf.split(model_output, num_or_size_splits=2, axis=0)
              model_output=output_uncond+cfg_scale*(output_cond-output_uncond) # W*(op_cond-op_uncond)+op_uncond

            latent=sampler.step(timestep, latents, model_output)

          decoder=models["decoder"]
          image=decoder(latents)
          image=rescale(image, (-1,1), (0,255))
          image=tf.cast(image, tf.uint8)
          return image


def rescale(x, old_range, new_range, clamp=False):
  old_min, old_max=old_range
  new_min, new_max=new_range
  x=x*(new_max-new_min)/(old_max-old_min)+new_min
  if clamp:
    x=tf.clip_by_value(x, new_min, new_max)
  return x


def get_time_embedding(timestep):
  freqs=tf.pow(10000, -tf.range(0, 160, dtype=tf.float32)/160) ## Same as normal positional encoding
  x=tf.cast(tf.expand_dims([timestep], axis=-1), tf.float32)*tf.expand_dims(freqs, axis=0)
  return tf.concat([tf.cos(x), tf.sin(x)], axis=-1)
